# 4 — Statistical modelling

**Inputs:**

| File | Contents |
|---|---|
| `data/curated/model_table.parquet` | 26,256 airport-hours × 79 columns, both targets, the split, and every flag. |
| `data/curated/model_table_roles.json` | The feature list and the leakage contract, read rather than hardcoded. |

**Outputs:** `models/` (both fitted models and the coefficient table),
`data/curated/model_predictions.parquet`, `data/curated/model_results.json`, and three
figures in `plots/`.

## Two models with different jobs

| | Model 1 | Model 2 |
|---|---|---|
| Question | How many pickups will this airport-hour produce? | What will a pickup in it be worth? |
| Target | `n_pickups` (count) | `mean_total` (US$, continuous) |
| Family | Poisson GLM, log link, quasi-Poisson scale | Gradient-boosted regression trees |
| Nature | Parametric, interpretable as rate ratios | Non-parametric, captures interactions and non-linearity |
| Missing data | Needs a complete design matrix | Handles missing values natively |
| Benchmark | Seasonal naive: last week, same hour | Seasonal naive: last week, same hour |

They are not two algorithms racing on one target. Their **product** is expected revenue per
airport-hour, which is the surface the recommendations stand on: *queue at LaGuardia at
22:00* is only advice if those trips pay. Notebook 2c verified that
`n_pickups × mean_total = sum_total_amount` holds in the data to the cent, so multiplying
the two predictions estimates a quantity that exists rather than one that sounds plausible.

## What notebook 3 established, and what it changes here

1. **The target is overdispersed within airport-hour cells.** A Poisson GLM's point
   predictions survive that; its standard errors do not. The fit below therefore uses a
   quasi-Poisson scale, and a negative binomial with an estimated dispersion is fitted as a
   contrast to show the choice was tested rather than assumed.
2. **Volume and value do not peak in the same cells.** That is the empirical warrant for
   modelling them separately.
3. **Arrivals lead pickups**, which is why the schedule features enter both at the current
   hour and lagged.
4. **Weather is a weak effect on volume.** Expect small coefficients and report them
   honestly rather than hunting for a specification that makes them significant.

## Rules this notebook holds itself to

- **Prediction runs forward in time.** Train is the calendar year 2023, test is January–June
  2024, and the split is read from the `split` column 2c materialised. No shuffled k-fold
  appears anywhere, including inside the tuning loop — the validation split used to choose
  the boosting hyper-parameters is the last quarter of the training year, not a random
  subset of it.
- **No feature may be a same-hour outcome.** The roles manifest lists every leaking column
  and the assertion in Step 2 fails the run if one reaches a feature list.
- **Standardisation and imputation are fitted on the training split alone**, then applied to
  the test split. Fitting them on the whole table would leak the test distribution into
  training, and it is invisible unless someone checks.

In [1]:
"""Fit, compare, and combine the two models.

Model 1 is a Poisson GLM with a quasi-Poisson scale for the pickup count;
Model 2 is a gradient-boosted tree ensemble for the mean fare. Their product is
expected revenue per airport-hour, which is the quantity the report's
recommendations are built on.

Reads the feature list and the leakage contract from `model_table_roles.json`
rather than hardcoding them, so that a change in notebook 2c propagates here
instead of silently diverging.
"""

import json
import sys
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import statsmodels.api as sm
import statsmodels.formula.api as smf
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.inspection import permutation_importance

sys.path.append(str(Path("..") / "scripts"))
from plot_utils import (  # noqa: E402
    BASELINE_COLOUR,
    DIVERGING_CMAP,
    PALETTE,
    airport_colour,
    currency_axis,
    heatmap,
    new_figure,
    save_figure,
    set_plot_style,
)

# --- Paths -----------------------------------------------------------------
# Notebook is expected to run from `notebooks/`.
PROJECT_ROOT = Path("..").resolve()
CURATED_DIR = PROJECT_ROOT / "data" / "curated"
MODELS_DIR = PROJECT_ROOT / "models"
PLOTS_DIR = PROJECT_ROOT / "plots"
for directory in (MODELS_DIR, PLOTS_DIR):
    directory.mkdir(parents=True, exist_ok=True)

# --- Reproducibility -------------------------------------------------------
SEED = 42

# --- The tuning split ------------------------------------------------------
# Hyper-parameters are chosen on the last quarter of the training year and the
# chosen configuration is refitted on the whole of it. This is a split in time,
# not a random one: the specification requires predictions to use future data,
# and that applies to model selection as much as to evaluation.
TUNE_VALID_START = "2023-10-01"

# Spark is not used in this notebook. The model table is 26,256 rows — small
# enough that a JVM would cost more than it saves — and both model libraries
# want a pandas frame. Everything upstream of this point was Spark precisely so
# that this step could be small.
set_plot_style()
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 40)

## Step 0 — Load the table and the contract

The roles manifest is the interface between 2c and this notebook. Reading the feature list
from it, rather than retyping twenty column names, means a feature added or renamed upstream
either appears here automatically or fails loudly.

In [2]:
table = pd.read_parquet(CURATED_DIR / "model_table.parquet")
table["date"] = pd.to_datetime(table["date"])

with open(CURATED_DIR / "model_table_roles.json") as handle:
    roles = json.load(handle)

assert len(table) == roles["rows"], (
    f"{len(table):,} rows read, manifest declares {roles['rows']:,}"
)

TARGET_VOLUME = roles["targets"]["model_1_volume"]
TARGET_VALUE = roles["targets"]["model_2_value"]
benchmark = roles["benchmark"]

AIRPORTS = tuple(sorted(table["airport"].unique()))

print(f"{len(table):,} rows × {len(table.columns)} columns")
print(f"Airports: {', '.join(AIRPORTS)}")
print(f"Targets: {TARGET_VOLUME} (Model 1), {TARGET_VALUE} (Model 2)")
print("\nSeasonal-naive benchmark to beat, fixed in 2c before any model was fitted:")
print(f"  {TARGET_VOLUME:<12} RMSE {benchmark['test_rmse_pickups']:>7.2f} pickups   "
      f"MAE {benchmark['test_mae_pickups']:>6.2f}")
print(f"  {TARGET_VALUE:<12} RMSE ${benchmark['test_rmse_mean_total']:>6.2f}   "
      f"MAE ${benchmark['test_mae_mean_total']:>5.2f}")

26,256 rows × 79 columns
Airports: JFK, LGA
Targets: n_pickups (Model 1), mean_total (Model 2)

Seasonal-naive benchmark to beat, fixed in 2c before any model was fitted:
  n_pickups    RMSE   52.84 pickups   MAE  36.08
  mean_total   RMSE $ 10.13   MAE $ 5.18


## Step 1 — The modelling set

Two exclusions, both flagged upstream rather than decided here:

- **`dst_anomaly`** — six rows. Two are the 02:00 that does not exist on the spring-forward
  dates, filled with zero pickups by the spine; one is the 01:00 that occurs twice in
  November and therefore contains two clock hours of trips. Neither is an observation of
  demand, and the November row would be read as an outlier by any model that saw it.
- **`has_full_lags` false** — the first week of 2023, whose weekly lag does not exist. These
  are excluded from the *fit*; notebook 3 used them, because a distribution should describe
  the whole window even when a model cannot use every row of it.

The count benchmark in 2c was computed on the test split excluding the daylight-saving
rows, which is exactly the row set Model 1 is scored on. The *value* benchmark is not:
it is undefined wherever the same hour last week was empty, so it covers fewer rows than
Model 2 can be scored on. Step 4 recomputes both on their common support rather than
quoting the manifest figure against a model evaluated on more rows than produced it.

In [3]:
usable = table[table["has_full_lags"] & ~table["dst_anomaly"]].copy()

train = usable[usable["split"] == "train"].sort_values(["airport", "date", "hour"])
test = usable[usable["split"] == "test"].sort_values(["airport", "date", "hour"])

excluded = len(table) - len(usable)
print(f"Excluded from the fit: {excluded} rows "
      f"({int(table['dst_anomaly'].sum())} daylight-saving, "
      f"{int((~table['has_full_lags']).sum())} without a weekly lag)")
print(f"train  {len(train):,} rows  ({train['date'].min().date()} to "
      f"{train['date'].max().date()})")
print(f"test   {len(test):,} rows  ({test['date'].min().date()} to "
      f"{test['date'].max().date()})")

# The two splits must not overlap in time, and every test date must follow
# every training date. Checked rather than trusted: this is the one property
# the whole evaluation rests on.
assert train["date"].max() < test["date"].min(), "The split is not forward in time"
assert len(train) + len(test) == len(usable)

# Model 2 is undefined where no trip occurred, so its support is smaller. The
# gap is reported here rather than discovered in a row count later.
value_train = train[train[TARGET_VALUE].notna()]
value_test = test[test[TARGET_VALUE].notna()]
print(f"\nModel 2 support: {len(value_train):,} train / {len(value_test):,} test "
      f"rows where {TARGET_VALUE} is defined "
      f"({len(train) - len(value_train)} and {len(test) - len(value_test)} "
      "empty hours dropped)")

Excluded from the fit: 342 rows (6 daylight-saving, 336 without a weekly lag)
train  17,180 rows  (2023-01-08 to 2023-12-31)
test   8,734 rows  (2024-01-01 to 2024-06-30)

Model 2 support: 16,375 train / 8,375 test rows where mean_total is defined (805 and 359 empty hours dropped)


## Step 2 — Features, and the leakage check

The manifest groups features by kind because the two models want different subsets:

- **Both models** take the calendar, the published arrival schedule, the Newark context, the
  weather, and the one-hour-lagged realised flight and taxi columns.
- **Model 1** additionally takes the volume history (`pickups_lag_*`).
- **Model 2** additionally takes the value history (`mean_total_lag_*`).

Neither takes the other's history. A model of what a trip is worth should not be handed the
count it is about to be multiplied by, or the product of the two predictions double-counts
one signal and the decomposition stops being interpretable.

The manifest's `cyclical` pair, `hour_sin` and `hour_cos`, is used by neither model. Step 3
says why: two harmonics cannot express LaGuardia's overnight curfew, and the GLM uses hour
dummies interacted with airport instead. A declared feature that goes unused is stated here
rather than left for a marker to notice.

`is_weekend` is dropped from the GLM only: it is an exact function of the day-of-week
dummies, so including both makes the design matrix rank-deficient. The trees keep it, since
a redundant split costs them nothing.

The assertion is the point of this cell. Every same-hour outcome — `mean_fare`,
`share_flat_fare`, `actual_arrivals`, `mean_arr_delay_min` — is listed under
`leak_if_unlagged` in the manifest, and if any of them reaches a feature list the run stops
here rather than reporting an excellent fit for having been told the answer.

In [4]:
groups = roles["features"]
leak_if_unlagged = set(roles["leak_if_unlagged"])

SHARED_GROUPS = [
    "temporal", "categorical", "flight_schedule", "flight_realised_lagged",
    "ewr", "weather", "taxi_lagged",
]


def collect(group_names: list) -> list:
    """Flatten feature groups into an ordered, de-duplicated list.

    ``temporal`` and ``categorical`` overlap by design in the manifest, so a
    plain concatenation would repeat ``day_of_week`` and ``month``.

    Args:
        group_names: Keys of the manifest's ``features`` mapping.

    Returns:
        Feature names, first occurrence order preserved.
    """
    seen, ordered = set(), []
    for name in group_names:
        for column in groups[name]:
            if column not in seen:
                seen.add(column)
                ordered.append(column)
    return ordered


MODEL_1_FEATURES = collect(SHARED_GROUPS + ["lags_volume"])
MODEL_2_FEATURES = collect(SHARED_GROUPS + ["lags_value"])

for name, features in [("Model 1", MODEL_1_FEATURES), ("Model 2", MODEL_2_FEATURES)]:
    leaking = sorted(set(features) & leak_if_unlagged)
    assert not leaking, f"{name} would use same-hour outcomes: {leaking}"
    absent = [column for column in features if column not in table.columns]
    assert not absent, f"{name} declares features absent from the table: {absent}"
    print(f"{name}: {len(features)} features")

print("\nNo same-hour outcome reaches either feature list.")
print(f"({len(leak_if_unlagged)} columns are barred by the manifest.)")

Model 1: 39 features
Model 2: 37 features

No same-hour outcome reaches either feature list.
(23 columns are barred by the manifest.)


## Step 3 — Model 1: a Poisson GLM for demand

### Why this model

The response is a non-negative integer count with a hard floor at zero and a strongly
right-skewed conditional distribution, which is what a Poisson GLM with a log link is for.
Ordinary least squares on the raw count would place mass below zero in exactly the overnight
hours notebook 3 showed are empty. The model is standard and covered in the prerequisite
subjects, so it is cited rather than introduced [1].

The log link earns its place in the recommendations, not just in the fit: a coefficient is a
**rate ratio**, so "18:00 carries e^β times the demand of 03:00" is a sentence a driver's
advisor can act on, which is not true of a black-box prediction of the same quantity.

### The design

- **Hour enters as 23 dummies, interacted with airport**, rather than as the cyclical terms
  the manifest also offers. Two harmonics can express one smooth daily cycle; they cannot
  express a hard overnight floor at one airport and not the other, and LaGuardia's curfew is
  exactly that. The interaction is what lets the two airports have different daily shapes,
  and its coefficients are read directly in Figure 9.
- **Day of week and month enter as dummies**, since neither is a quantity — Wednesday is not
  three of anything.
- **Continuous features are standardised** on the training split, so a coefficient is the
  effect of a one-standard-deviation move and the flight, weather, and lag blocks are
  comparable to each other.

### Assumptions, stated before the fit

1. Observations are conditionally independent given the features. Airport-hours are
   autocorrelated, which the lag features absorb rather than eliminate; the residual
   autocorrelation is checked in Step 5.
2. The variance is proportional to the mean. Notebook 3 established that the ratio is above
   one, so the **quasi-Poisson** scale is used: the Pearson chi-squared statistic divided by
   its degrees of freedom multiplies the standard errors. Point predictions are unaffected;
   inference is.
3. No same-hour outcome is in the design, enforced in Step 2.

### Imputation, fitted on training data only

A GLM needs a complete design matrix, so three families of null have to be resolved. Each is
structural, and none is filled with a guess about the outcome:

| Family | Columns | Fill | Why |
|---|---|---|---|
| Nothing scheduled to land | `share_longhaul`, the lagged cancellation and delay shares | 0 | The share of an empty set. `sched_arrivals = 0` in the same row tells the model the fill is structural. |
| Window edge | `sched_arrivals_prev_hr`, `sched_arrivals_next_hr` | 0 | The hour before the first hour of the window is outside it. Two rows. |
| Weather gap | `temp_c`, `rhum_pct`, `wspd_kmh`, `pres_hpa` | Training median | Beyond 2c's carry-forward limit. Small, and flagged in the table as `weather_missing`. |

In [5]:
# Filled with zero because the quantity is a statistic of an empty set, not
# because the value is unknown.
STRUCTURAL_ZERO = [
    "share_longhaul",
    "sched_arrivals_prev_hr",
    "sched_arrivals_next_hr",
    "share_cancelled_lag_1h",
    "share_delayed_15_lag_1h",
    "mean_arr_delay_min_lag_1h",
    "n_delay_observed_lag_1h",
    "arrival_slippage_lag_1h",
    "actual_arrivals_lag_1h",
    "ewr_share_cancelled_lag_1h",
    "ewr_mean_arr_delay_min_lag_1h",
    "ewr_n_delay_observed_lag_1h",
]

# Filled with the training median because the measurement exists but was not
# recorded, and the quantity is smooth.
MEDIAN_FILL = ["temp_c", "rhum_pct", "wspd_kmh", "pres_hpa", "prcp_mm"]

# Everything the taxi-side one-hour lags describe is undefined for an hour that
# had no trips, which is a statement about demand rather than about the fare.
LAG_MEDIAN_FILL = [
    "mean_fare_lag_1h", "share_flat_fare_lag_1h", "mean_distance_mi_lag_1h",
    "mean_tip_ratio_lag_1h", "share_flex_fare_lag_1h", "mean_total_lag_168h",
    "mean_total_same_hour_7d",
]

BOOLEAN_FEATURES = [
    "is_weekend", "is_holiday", "is_holiday_adjacent", "is_precip", "is_freezing",
]


def fit_imputer(frame: pd.DataFrame, columns: list) -> dict:
    """Learn median fill values from the training split.

    Args:
        frame: Training rows only. Passing the full table here would leak the
            test distribution into the fill.
        columns: Columns to learn a median for.

    Returns:
        Mapping from column name to the training median.
    """
    return {
        column: float(frame[column].median())
        for column in columns
        if column in frame.columns
    }


def apply_imputation(
    frame: pd.DataFrame, medians: dict, features: list
) -> pd.DataFrame:
    """Resolve the documented null families in a copy of ``frame``.

    Args:
        frame: Rows to impute.
        medians: Fill values from :func:`fit_imputer`.
        features: Feature columns that must be complete afterwards.

    Returns:
        A copy with no nulls in ``features``.

    Raises:
        AssertionError: If any feature still holds a null, which means a null
            family reached this notebook without being documented.
    """
    filled = frame.copy()
    for column in STRUCTURAL_ZERO:
        if column in filled.columns:
            filled[column] = filled[column].fillna(0.0)
    for column, value in medians.items():
        filled[column] = filled[column].fillna(value)
    for column in BOOLEAN_FEATURES:
        if column in filled.columns:
            filled[column] = filled[column].fillna(False).astype(bool)

    remaining = [
        column for column in features if filled[column].isna().any()
    ]
    assert not remaining, f"Undocumented nulls survive imputation in: {remaining}"
    return filled


medians = fit_imputer(train, MEDIAN_FILL + LAG_MEDIAN_FILL)
glm_train = apply_imputation(train, medians, MODEL_1_FEATURES)
glm_test = apply_imputation(test, medians, MODEL_1_FEATURES)

print(f"Median fills learned from {len(train):,} training rows only:")
for column, value in list(medians.items())[:5]:
    print(f"  {column:<26} {value:.2f}")
print(f"  ... {len(medians)} columns in total")

Median fills learned from 17,180 training rows only:
  temp_c                     13.90
  rhum_pct                   64.00
  wspd_kmh                   16.60
  pres_hpa                   1015.70
  prcp_mm                    0.00
  ... 12 columns in total


In [6]:
# Standardisation, on the training split alone. Binary and categorical columns
# are left alone: a standardised dummy is harder to read and no better behaved.
CATEGORICAL = ["hour", "day_of_week", "month", "airport"]
CONTINUOUS = [
    column for column in MODEL_1_FEATURES
    if column not in CATEGORICAL + BOOLEAN_FEATURES
]

scaler = {
    column: (float(glm_train[column].mean()), float(glm_train[column].std(ddof=0)))
    for column in CONTINUOUS
}


def standardise(frame: pd.DataFrame, statistics: dict) -> pd.DataFrame:
    """Centre and scale continuous columns with training statistics.

    Args:
        frame: Rows to transform.
        statistics: Mapping from column to ``(mean, standard deviation)``,
            learned on the training split.

    Returns:
        A copy in which each continuous column is expressed in training
        standard deviations. A column with no variation in training is left
        centred but unscaled rather than divided by zero.
    """
    scaled = frame.copy()
    for column, (centre, spread) in statistics.items():
        scaled[column] = (scaled[column] - centre) / (spread if spread > 0 else 1.0)
    return scaled


glm_train = standardise(glm_train, scaler)
glm_test = standardise(glm_test, scaler)

# `is_weekend` is an exact function of the day-of-week dummies and would make
# the design matrix rank-deficient.
GLM_TERMS = (
    ["C(hour) * airport", "C(day_of_week)", "C(month)"]
    + ["is_holiday", "is_holiday_adjacent", "is_precip", "is_freezing"]
    + CONTINUOUS
)
FORMULA = f"{TARGET_VOLUME} ~ " + " + ".join(GLM_TERMS)

print(f"{len(CONTINUOUS)} continuous features standardised on the training split")
print(f"Formula: {TARGET_VOLUME} ~ C(hour) * airport + C(day_of_week) + C(month) "
      f"+ 4 indicators + {len(CONTINUOUS)} standardised continuous terms")

30 continuous features standardised on the training split
Formula: n_pickups ~ C(hour) * airport + C(day_of_week) + C(month) + 4 indicators + 30 standardised continuous terms


In [7]:
# Use parameter convergence for both fits so the Pearson scale cannot
# change the stopping criterion.

poisson_fit = smf.glm(
    formula=FORMULA,
    data=glm_train,
    family=sm.families.Poisson()
).fit(
    tol_criterion="params",
    atol=1e-10,
    rtol=1e-10,
    maxiter=200
)

# Overdispersion measured from the ordinary Poisson fit.
pearson_dispersion = float(
    poisson_fit.pearson_chi2 / poisson_fit.df_resid
)

# Same mean model, but Pearson/quasi-Poisson scale for inference.
quasi_fit = smf.glm(
    formula=FORMULA,
    data=glm_train,
    family=sm.families.Poisson()
).fit(
    scale="X2",
    tol_criterion="params",
    atol=1e-10,
    rtol=1e-10,
    maxiter=200
)

print(f"Parameters estimated:        {int(poisson_fit.df_model) + 1}")
print(f"Residual degrees of freedom: {int(poisson_fit.df_resid):,}")
print(f"Pearson dispersion:          {pearson_dispersion:.2f}")
print(
    f"Standard errors inflated by: "
    f"{np.sqrt(pearson_dispersion):.2f}x under the quasi-Poisson scale"
)

max_coef_diff = np.max(
    np.abs(poisson_fit.params - quasi_fit.params)
)

print(f"Maximum coefficient difference: {max_coef_diff:.3e}")

assert np.allclose(
    poisson_fit.params,
    quasi_fit.params,
    rtol=1e-8,
    atol=1e-10,
), (
    "Poisson and quasi-Poisson coefficient estimates differ "
    "beyond numerical tolerance"
)

Parameters estimated:        98
Residual degrees of freedom: 17,082
Pearson dispersion:          10.48
Standard errors inflated by: 3.24x under the quasi-Poisson scale
Maximum coefficient difference: 0.000e+00


### The negative binomial contrast

The quasi-Poisson corrects the standard errors but keeps the Poisson's variance shape,
`Var = φ·μ`. A negative binomial assumes `Var = μ + α·μ²` instead — dispersion that grows
with the square of the mean, which is the pattern Figure 2 of notebook 3 actually shows.

α is estimated by the standard auxiliary regression [2]: regress
`((y − μ̂)² − y) / μ̂` on `μ̂` through the origin, using the Poisson fitted values. The
result is then used to fit a negative binomial GLM on the same design, so the two count
models differ in exactly one assumption and nothing else.

If the two agree on the test set, the report can say the demand model is robust to the
variance assumption, which is a stronger statement than either fit alone. They are
compared on likelihood as well as on test error; the negative binomial's AIC is charged
for the α estimated above, which statsmodels treats as known.

In [ ]:
mu = np.asarray(poisson_fit.fittedvalues, dtype=float)
observed = np.asarray(glm_train[TARGET_VOLUME], dtype=float)

# NB2: Var = mu + alpha * mu^2, so ((y - mu)^2 - y) / mu = alpha * mu + error.
# Fitted through the origin, on the training split only.
auxiliary = ((observed - mu) ** 2 - observed) / mu
alpha = float(sm.OLS(auxiliary, mu).fit().params[0])

negbin_fit = smf.glm(
    formula=FORMULA,
    data=glm_train,
    family=sm.families.NegativeBinomial(alpha=max(alpha, 1e-6)),
).fit()

if alpha <= 0:
    print(f"Estimated alpha is {alpha:.4f}, at or below zero: the auxiliary "
          "regression finds no dispersion beyond the Poisson's own, so the "
          "negative binomial below collapses to it. Report that, rather than "
          "the fitted alpha.")
print(f"Estimated dispersion alpha: {alpha:.4f}")
print(f"Poisson log-likelihood:           {poisson_fit.llf:,.0f}")
print(f"Negative binomial log-likelihood: {negbin_fit.llf:,.0f}")
# statsmodels treats a fixed `alpha` as known and does not charge the AIC for
# it. It was estimated from these data by the auxiliary regression above, so it
# costs a parameter and the AIC is corrected by two. The comparison does not
# turn on the correction — the gap is four orders of magnitude larger — but an
# uncorrected figure would flatter the model that was given an extra degree of
# freedom.
poisson_aic = float(poisson_fit.aic)
negbin_aic = float(negbin_fit.aic) + 2.0
print(f"Poisson AIC:           {poisson_aic:,.0f}")
print(f"Negative binomial AIC: {negbin_aic:,.0f}  (alpha charged as estimated)")

## Step 4 — Model 2: gradient-boosted trees for the fare

### Why a contrasting model rather than a second GLM

Model 1 buys interpretability with a strong functional assumption: every effect is additive
in the log rate, and only the interactions written into the formula exist. Model 2 is
chosen to fail differently.

- The fare paid at an airport-hour is a **mixture**: JFK flat fares to Manhattan, metered
  runs into Queens, and out-of-city rate codes, in proportions that shift with the hour and
  the day. A mixture whose weights move is exactly the case where an additive model
  underfits and a tree ensemble does not, because a tree can condition on hour *and*
  airport *and* the arrival schedule simultaneously without being told to.
- Trees are **scale-free and handle missing values natively**, so Model 2 needs neither the
  standardisation nor the imputation Model 1 required. That difference is not a convenience
  — it means the two models see the data differently, and agreement between them is
  evidence rather than an artefact of a shared preprocessing decision.

The cost is interpretability, which is recovered through permutation importance rather than
claimed to be unnecessary. Gradient boosting is not assumed to be covered by the
prerequisite subjects, so it is stated with a reference [3]: an additive ensemble of shallow
regression trees, each fitted to the gradient of the squared-error loss left by the ones
before it, with the learning rate controlling how much of each correction is taken.

### Tuning without looking forward

Hyper-parameters are chosen on **October to December 2023** and the winning configuration is
refitted on the whole training year. Two consequences of validating in time rather than at
random are worth stating, because both are silent by default:

- The validation quarter's months are months the tuning fold never saw. The shortfall falls
  equally on every candidate, so the comparison between configurations still holds, and the
  refit sees all twelve.
- Categorical levels are therefore frozen from the training split rather than inferred per
  frame. A frame typed on its own would give the validation quarter the levels
  `{10, 11, 12}` and the tuning fold `{1, …, 9}`, and scikit-learn maps categoricals by
  label — every validation row's month would be routed to the unknown branch without a
  warning. A random validation split would let the model be tuned
on a July afternoon in order to predict a February morning — the same objection the
specification raises against shuffled k-fold, applied one level up. The test set is not
touched at any point in this step.

In [ ]:
# The categorical levels are frozen once, from the training split, and reused
# for every frame the tree model ever sees.
#
# This is not housekeeping. ``astype("category")`` takes its levels from
# whichever frame it is handed, and scikit-learn maps a categorical column by
# its level labels. The tuning split below trains on January-September and
# validates on October-December, so typing each frame independently would give
# the validation quarter the level set {10, 11, 12} and the training fold
# {1, ..., 9} — every validation row's month silently routed to the unknown
# branch. Freezing the levels makes the encoding identical everywhere, and the
# assertion inside `design_matrix` turns a genuinely unseen level into a failure
# rather than a shrug.
CATEGORY_LEVELS = {
    column: pd.CategoricalDtype(categories=sorted(train[column].unique()))
    for column in CATEGORICAL
}


def design_matrix(frame: pd.DataFrame, features: list) -> pd.DataFrame:
    """Build the tree model's feature frame.

    No imputation and no scaling: ``HistGradientBoostingRegressor`` splits on
    missingness directly, and trees are invariant to monotone rescaling of a
    feature. Categorical columns are typed as such so the implementation
    partitions their levels rather than treating the codes as magnitudes —
    month 12 is not twelve of anything.

    Args:
        frame: Rows to transform.
        features: Feature columns to include.

    Returns:
        A frame of features, categoricals typed against the frozen training
        levels and booleans cast to float so that a missing indicator survives
        as NaN rather than raising.

    Raises:
        AssertionError: If a categorical column holds a level absent from the
            training split, which would otherwise be encoded as unknown
            without comment.
    """
    matrix = frame[features].copy()
    for column in CATEGORICAL:
        if column in matrix.columns:
            levels = CATEGORY_LEVELS[column]
            unseen = sorted(
                set(matrix[column].dropna()) - set(levels.categories)
            )
            assert not unseen, (
                f"{column} holds levels the training split never saw: {unseen}"
            )
            matrix[column] = matrix[column].astype(levels)
    for column in BOOLEAN_FEATURES:
        if column in matrix.columns:
            matrix[column] = matrix[column].map({True: 1.0, False: 0.0}).astype(float)
    return matrix


x_value_train = design_matrix(value_train, MODEL_2_FEATURES)
y_value_train = value_train[TARGET_VALUE].to_numpy()
x_value_test = design_matrix(value_test, MODEL_2_FEATURES)
y_value_test = value_test[TARGET_VALUE].to_numpy()

missing_share = 100 * x_value_train.isna().to_numpy().mean()
print(f"Model 2 design: {x_value_train.shape[0]:,} rows × "
      f"{x_value_train.shape[1]} features")
print(f"{missing_share:.2f}% of feature cells are missing and are passed to the "
      "model as such, unimputed")

In [ ]:
def rmse(actual, predicted) -> float:
    """Root mean squared error, ignoring rows where either value is missing."""
    actual = np.asarray(actual, dtype=float)
    predicted = np.asarray(predicted, dtype=float)
    usable = ~(np.isnan(actual) | np.isnan(predicted))
    return float(np.sqrt(np.mean((actual[usable] - predicted[usable]) ** 2)))


def error_metrics(actual, predicted) -> dict:
    """Root mean squared error and mean absolute error.

    Args:
        actual: Observed values.
        predicted: Predicted values.

    Returns:
        Mapping with ``n``, ``rmse``, and ``mae``, computed over the rows where
        both are present.
    """
    actual = np.asarray(actual, dtype=float)
    predicted = np.asarray(predicted, dtype=float)
    usable = ~(np.isnan(actual) | np.isnan(predicted))
    error = actual[usable] - predicted[usable]
    return {
        "n": int(usable.sum()),
        "rmse": round(float(np.sqrt(np.mean(error ** 2))), 3),
        "mae": round(float(np.mean(np.abs(error))), 3),
    }


# A deliberately small grid. The point is to show the configuration was chosen
# on held-out future data rather than to squeeze out a last decimal place, and
# every extra configuration is another chance to overfit the validation
# quarter.
GRID = [
    {"learning_rate": 0.05, "max_leaf_nodes": 15, "max_iter": 300},
    {"learning_rate": 0.05, "max_leaf_nodes": 31, "max_iter": 300},
    {"learning_rate": 0.05, "max_leaf_nodes": 31, "max_iter": 600},
    {"learning_rate": 0.10, "max_leaf_nodes": 15, "max_iter": 200},
    {"learning_rate": 0.10, "max_leaf_nodes": 31, "max_iter": 200},
    {"learning_rate": 0.10, "max_leaf_nodes": 63, "max_iter": 200},
]

tune_cutoff = pd.Timestamp(TUNE_VALID_START)
inner_train = value_train[value_train["date"] < tune_cutoff]
inner_valid = value_train[value_train["date"] >= tune_cutoff]
assert inner_train["date"].max() < inner_valid["date"].min()

# A forward validation split means the validation quarter's months are, by
# construction, months the tuning fold never saw. With the levels frozen above
# they are at least encoded consistently, and the shortfall is the same for
# every candidate, so the comparison between configurations still holds. Stated
# here because it is a real limitation of tuning in time rather than at random,
# and the final refit below does see all twelve months.
unseen_months = sorted(set(inner_valid["month"]) - set(inner_train["month"]))
print(f"Months absent from the tuning fold: {unseen_months} — inherent to a "
      "forward split; the refit uses the whole year")

print(f"Tuning on {len(inner_train):,} rows to "
      f"{inner_train['date'].max().date()}, validating on "
      f"{len(inner_valid):,} rows from {inner_valid['date'].min().date()}")

x_inner_train = design_matrix(inner_train, MODEL_2_FEATURES)
x_inner_valid = design_matrix(inner_valid, MODEL_2_FEATURES)

results = []
for parameters in GRID:
    candidate = HistGradientBoostingRegressor(
        random_state=SEED, early_stopping=False, **parameters
    )
    candidate.fit(x_inner_train, inner_train[TARGET_VALUE].to_numpy())
    score = rmse(
        inner_valid[TARGET_VALUE].to_numpy(), candidate.predict(x_inner_valid)
    )
    results.append({**parameters, "valid_rmse": round(score, 4)})

tuning = pd.DataFrame(results).sort_values("valid_rmse").reset_index(drop=True)
print("\nValidation RMSE on the last quarter of 2023 (US$ per trip)")
print(tuning.to_string(index=False))

best = {
    "learning_rate": float(tuning.loc[0, "learning_rate"]),
    "max_leaf_nodes": int(tuning.loc[0, "max_leaf_nodes"]),
    "max_iter": int(tuning.loc[0, "max_iter"]),
}
print(f"\nChosen: {best}")

In [ ]:
# Refit the chosen configuration on the whole training year.
gbt = HistGradientBoostingRegressor(
    random_state=SEED, early_stopping=False, **best
)
gbt.fit(x_value_train, y_value_train)
gbt_test_prediction = gbt.predict(x_value_test)

# --- Scoring the benchmark on the rows it actually covers -------------------
# The manifest's value benchmark was computed in 2c wherever both `mean_total`
# and `mean_total_lag_168h` are defined. Model 2 is defined on more rows than
# that: an hour whose counterpart last week was empty has no naive forecast,
# but the tree model predicts it without complaint. Quoting one against the
# other would set a model scored on every hour beside a benchmark scored on the
# easier subset, and the hours it drops are the sparse ones.
#
# Both are therefore scored on their common support, and Model 2's full-support
# figure is reported alongside rather than quietly discarded.
naive_value = value_test["mean_total_lag_168h"].to_numpy(dtype=float)
common = ~pd.isna(naive_value)

value_metrics = {
    "benchmark": error_metrics(y_value_test[common], naive_value[common]),
    "gbt": error_metrics(y_value_test[common], gbt_test_prediction[common]),
    "gbt_full_support": error_metrics(y_value_test, gbt_test_prediction),
}

# Recomputed rather than read from the manifest, so that a change in 2c fails
# here instead of being quoted stale.
assert abs(
    value_metrics["benchmark"]["rmse"] - benchmark["test_rmse_mean_total"]
) < 0.01, (
    f"Recomputed value benchmark {value_metrics['benchmark']['rmse']} differs "
    f"from the manifest's {benchmark['test_rmse_mean_total']}"
)
assert value_metrics["benchmark"]["n"] == value_metrics["gbt"]["n"], (
    "The value benchmark and Model 2 are not scored on the same rows"
)

improvement = 100 * (
    1 - value_metrics["gbt"]["rmse"] / value_metrics["benchmark"]["rmse"]
)
dropped = len(value_test) - value_metrics["gbt"]["n"]

print(f"Model 2 on the test period, common support: "
      f"{value_metrics['gbt']['n']:,} of {len(value_test):,} rows where "
      f"{TARGET_VALUE} is defined")
print(f"  seasonal naive   RMSE ${value_metrics['benchmark']['rmse']:.2f}   "
      f"MAE ${value_metrics['benchmark']['mae']:.2f}")
print(f"  boosted trees    RMSE ${value_metrics['gbt']['rmse']:.2f}   "
      f"MAE ${value_metrics['gbt']['mae']:.2f}")
print(f"  improvement in RMSE: {improvement:+.1f}%")
print(f"\nThe {dropped} rows the benchmark cannot score are hours whose "
      "counterpart last week was empty. Model 2 scores them at RMSE "
      f"${value_metrics['gbt_full_support']['rmse']:.2f} / MAE "
      f"${value_metrics['gbt_full_support']['mae']:.2f} over all "
      f"{value_metrics['gbt_full_support']['n']:,} rows, which is the figure to "
      "quote when no benchmark is being quoted beside it.")


## Step 5 — Comparison and error analysis

Three questions, in the order they matter to the report.

**Does either model beat the benchmark?** *This hour will be what it was last week* is a
strong, free predictor of airport demand. A model that does not beat it is not a result to
bury; it is a finding about how much of airport demand is pure weekly seasonality.

**Do the two count models agree?** The Poisson and the negative binomial differ in exactly
one assumption. If their test errors are close, the demand model is robust to the variance
assumption and the quasi-Poisson standard errors can be reported with confidence.

**Where does the error concentrate, and why?** Mean error by hour of day exposes whether a
model is systematically wrong somewhere a driver would act on it — an overprediction at
04:00 is a recommendation to sit in an empty queue, and matters far more than the same
error at 18:00. The residual autocorrelation reported here is the check on the independence
assumption listed in Step 3.

The residuals turn out to be mostly one-signed, which admits two explanations: a level shift
between the two years, or attenuation towards the middle of the fitted range. They are
distinguishable — a level shift moves the whole demand range the same way and must match
the sign of the year-on-year change, while attenuation is monotone in the predicted level
and changes sign in the middle of it — so the cell below tests both and derives the verdict
from the numbers instead of asserting one.

In [ ]:
predictions = test.copy()
predictions["pred_pickups"] = np.asarray(poisson_fit.predict(glm_test), dtype=float)
predictions["pred_pickups_nb"] = np.asarray(negbin_fit.predict(glm_test), dtype=float)
predictions["pred_mean_total"] = gbt.predict(design_matrix(test, MODEL_2_FEATURES))

# The count benchmark needs no restriction: `pickups_lag_168h` is defined for
# every row that has a full set of lags, so all three count rows below are
# scored on the same 8,734 hours. Recomputed here, and checked against 2c.
naive_pickups = predictions["pickups_lag_168h"].to_numpy(dtype=float)
assert not pd.isna(naive_pickups).any(), (
    "The seasonal-naive count forecast is undefined somewhere in the test split"
)

volume_metrics = {
    "benchmark": error_metrics(predictions[TARGET_VOLUME], naive_pickups),
    "poisson": error_metrics(
        predictions[TARGET_VOLUME], predictions["pred_pickups"]
    ),
    "negative_binomial": error_metrics(
        predictions[TARGET_VOLUME], predictions["pred_pickups_nb"]
    ),
}

assert abs(
    volume_metrics["benchmark"]["rmse"] - benchmark["test_rmse_pickups"]
) < 0.01, (
    f"Recomputed count benchmark {volume_metrics['benchmark']['rmse']} differs "
    f"from the manifest's {benchmark['test_rmse_pickups']}"
)
assert (
    volume_metrics["benchmark"]["n"]
    == volume_metrics["poisson"]["n"]
    == volume_metrics["negative_binomial"]["n"]
), "The count models and their benchmark are not scored on the same rows"


def comparison_row(target: str, label: str, metrics: dict) -> dict:
    """One row of the comparison table, benchmark or model alike.

    ``n`` is carried into the table rather than left in a footnote. Two of
    these rows are scored on a different number of hours from the other three,
    and a table that hides that invites exactly the comparison it should not
    support.
    """
    return {
        "target": target,
        "model": label,
        "n": int(metrics["n"]),
        "rmse": round(float(metrics["rmse"]), 3),
        "mae": round(float(metrics["mae"]), 3),
    }


comparison = pd.DataFrame([
    comparison_row("n_pickups", "seasonal naive (benchmark)",
                   volume_metrics["benchmark"]),
    comparison_row("n_pickups", "Poisson GLM", volume_metrics["poisson"]),
    comparison_row("n_pickups", "negative binomial GLM",
                   volume_metrics["negative_binomial"]),
    comparison_row("mean_total", "seasonal naive (benchmark)",
                   value_metrics["benchmark"]),
    comparison_row("mean_total", "boosted trees", value_metrics["gbt"]),
    comparison_row("mean_total", "boosted trees, full support",
                   value_metrics["gbt_full_support"]),
])
print("Test-period performance (January–June 2024). Rows are comparable only "
      "within a target and\nonly at equal n: the last row is the value model "
      "on hours the benchmark cannot score.")
print(comparison.to_string(index=False))

count_gap = abs(
    volume_metrics["poisson"]["rmse"] - volume_metrics["negative_binomial"]["rmse"]
)
relative_gap = count_gap / volume_metrics["poisson"]["rmse"]

# The verdict follows the number rather than being asserted regardless of it.
if relative_gap < 0.05:
    verdict = "the demand estimate does not hinge on the variance assumption"
else:
    verdict = (
        "the variance assumption does move the point predictions. A negative "
        "binomial weights each observation by mu / (1 + alpha * mu), which "
        "discounts the busiest hours relative to a Poisson — and the busiest "
        "hours are where squared error is decided. It fits the dispersion "
        "better and predicts the level slightly worse; report both, and use "
        "the Poisson for the recommendations"
    )
print(f"\nThe two count models differ by {count_gap:.2f} pickups in test RMSE, "
      f"{100 * relative_gap:.0f}% of the Poisson's — {verdict}.")

In [ ]:
predictions["residual"] = (
    predictions[TARGET_VOLUME] - predictions["pred_pickups"]
)

by_hour = (
    predictions.groupby(["airport", "hour"])["residual"]
    .agg(mean_error="mean", mean_abs_error=lambda column: column.abs().mean())
    .reset_index()
)

worst_hours = by_hour.reindex(
    by_hour["mean_error"].abs().sort_values(ascending=False).index
).head(6)
print("Hours where the demand model is most biased (observed minus predicted)")
print(worst_hours.round(1).to_string(index=False))

# Residual autocorrelation at one hour, within airport, over the ordered test
# split. Independence was assumed in Step 3; this is the check.
print("\nLag-1 residual autocorrelation on the test split")
for airport, group in predictions.groupby("airport"):
    ordered = group.sort_values(["date", "hour"])["residual"]
    print(f"  {airport}: {ordered.autocorr(lag=1):+.3f}")

# --- Attributing the one-signed bias ---------------------------------------
# The residuals are mostly one-signed. There are two candidate explanations and
# they are distinguishable, so both are tested rather than one being assumed.
#
#   (a) A level shift between the two years. The month dummies carry the season
#       within a year but cannot carry a change of level across years, and the
#       lag features only partly absorb it.
#   (b) Attenuation. A fit shrinks towards the middle of its range, so it
#       under-states the busy hours and over-states the quiet ones.
#
# They predict different things. A level shift moves every part of the demand
# range the same way, and its sign must match the sign of the year-on-year
# change. Attenuation is monotone in the predicted level and changes sign in
# the middle of it. Deciles of predicted demand separate the two.
mean_bias = float(predictions["residual"].mean())
direction = "under" if mean_bias > 0 else "over"
same_sign_cells = float((by_hour["mean_error"] > 0).mean())
print(f"\nMean residual on the test split: {mean_bias:+.1f} pickups per hour, "
      f"so the model {direction}-predicts on average; "
      f"{100 * same_sign_cells:.0f}% of the {len(by_hour)} (airport, hour) "
      "cells share that sign.")

# (a) The level, on the same calendar months in each year. The training year
#     runs to December and airport demand is seasonal, so comparing all of 2023
#     against January-June 2024 would report the summer peak as a change.
TEST_MONTHS = sorted(test["month"].unique())
level = pd.DataFrame({
    "train_same_months": (
        train[train["month"].isin(TEST_MONTHS)]
        .groupby("airport")[TARGET_VOLUME].mean()
    ),
    "test_mean": test.groupby("airport")[TARGET_VOLUME].mean(),
})
level["year_on_year_pct"] = 100 * (
    level["test_mean"] / level["train_same_months"] - 1
)
print("\nDemand level, same calendar months in each year")
print(level.round(2).to_string())

# (b) The bias against the predicted level.
by_decile = (
    predictions
    .assign(decile=pd.qcut(
        predictions["pred_pickups"], 10, labels=False, duplicates="drop"
    ) + 1)
    .groupby("decile")
    .agg(
        mean_predicted=("pred_pickups", "mean"),
        mean_observed=(TARGET_VOLUME, "mean"),
        mean_error=("residual", "mean"),
    )
)
print("\nMean residual by decile of predicted demand (pickups)")
print(by_decile.round(1).to_string())

# The verdict is computed from the two diagnostics rather than written in
# advance. An earlier version of this cell attributed the bias to the
# year-on-year change while printing a year-on-year change of the opposite
# sign, which is the failure this block exists to prevent.
level_shift_pct = float(level["year_on_year_pct"].mean())
signs_agree = (level_shift_pct > 0) == (mean_bias > 0)
decile_trend = float(
    by_decile["mean_error"].corr(by_decile["mean_predicted"], method="spearman")
)
decile_spread = float(
    by_decile["mean_error"].iloc[-1] - by_decile["mean_error"].iloc[0]
)

print()
if signs_agree:
    print(f"Demand moved {level_shift_pct:+.1f}% year on year and the model "
          f"{direction}-predicts: the signs agree, so part of the bias is a "
          "level the calendar dummies cannot carry across years.")
else:
    print(f"Demand moved {level_shift_pct:+.1f}% year on year while the model "
          f"{direction}-predicts. The signs disagree, so the bias is not a "
          "year-on-year level effect — that effect pushes the residuals the "
          "other way, and the bias survives it.")

if abs(decile_trend) > 0.6:
    print(f"The bias is monotone in the predicted level (rank correlation "
          f"{decile_trend:+.2f}), spanning {decile_spread:+.0f} pickups from "
          "the quietest decile to the busiest. That is attenuation: the fit is "
          "shrunk towards the middle of its range, so it under-states the busy "
          "hours a driver would queue for and over-states the empty ones. It "
          "is a shape error, not a level error, and it is the honest reading "
          "of Figure 8.")
else:
    print(f"The bias is flat across deciles of predicted demand (rank "
          f"correlation {decile_trend:+.2f}), which reads as a level effect "
          "rather than attenuation.")

worst_days = (
    predictions.assign(absolute_error=predictions["residual"].abs())
    .groupby("date")["absolute_error"].mean()
    .sort_values(ascending=False)
    .head(5)
)
print("\nDays with the largest mean absolute error")
for date, value in worst_days.items():
    names = predictions.loc[predictions["date"] == date, "holiday_name"].dropna()
    label = names.iloc[0] if len(names) else "—"
    print(f"  {date.date()}  {value:6.1f} pickups   {label}")


In [ ]:
# --- Figure 8 --------------------------------------------------------------
fig, axes = new_figure(1, 2, height=2.9)

ax = axes[0]
for airport, group in predictions.groupby("airport"):
    ax.scatter(group["pred_pickups"], group[TARGET_VOLUME], s=4, alpha=0.25,
               linewidths=0, color=airport_colour(airport), label=airport)
limit = float(
    max(predictions["pred_pickups"].max(), predictions[TARGET_VOLUME].max())
) * 1.05
ax.plot([0, limit], [0, limit], color=BASELINE_COLOUR, lw=1.0, ls="--",
        label="perfect prediction")

# Observed mean within twenty bins of predicted demand. The scatter alone shows
# spread; this line shows where the spread is off-centre, which is the claim
# the error analysis makes and the reader should be able to see rather than
# take on trust.
binned = (
    predictions
    .groupby(
        pd.qcut(predictions["pred_pickups"], 20, duplicates="drop"),
        observed=True,
    )
    .agg(x=("pred_pickups", "mean"), y=(TARGET_VOLUME, "mean"))
)
ax.plot(binned["x"], binned["y"], color=PALETTE["green"], lw=1.4,
        label="binned mean")
ax.set_xlim(0, limit)
ax.set_ylim(0, limit)
ax.set_xlabel("Predicted pickups")
ax.set_ylabel("Observed pickups")
ax.set_title("(a) Model 1 on the test period")
ax.legend(loc="upper left", markerscale=3)

ax = axes[1]
for airport, group in by_hour.groupby("airport"):
    ax.plot(group["hour"], group["mean_error"], marker="o", markersize=3,
            color=airport_colour(airport), label=airport)
ax.axhline(0, color=BASELINE_COLOUR, lw=0.8)
ax.set_xticks(range(0, 24, 3))
ax.set_xlabel("Hour of day")
ax.set_ylabel("Observed minus predicted")
ax.set_title("(b) Where the demand model is biased")
ax.legend()

fig.tight_layout()
save_figure(fig, "fig08_model1_diagnostics")

### Reading the two models

**Model 1 is read as an expected day.** Every continuous feature is set to its training mean
*within the hour being predicted*, every indicator is off, and the day is an ordinary
Wednesday in June — a month inside the test window, named in the caption rather than left
implicit; the fitted rate for each `(hour, airport)` is then what the model expects that
hour to bring on a normal midweek day, and the gap between the two curves is the interaction
term in the formula made visible.

The alternative — holding every feature at its overall mean — sounds more like a
*ceteris paribus* effect and is worse here. Four of the features are the demand history, and
fixing those at the daily average describes an hour that does not occur: an 06:00 preceded by
an average hour rather than by a nearly empty 05:00. Done that way the profile peaks at
06:00 and flattens the evening, which contradicts Figure 3 of notebook 3. The hour dummies in
this model carry the daily cycle *net of* what the lags explain, so they are not interpretable
on their own, and the report should not present them as such.

**Model 2 is read by permutation importance.** A feature is shuffled on the test set and the
increase in RMSE is recorded: features the model actually relies on degrade it, features it
ignores do not. This is measured on the test split, so it describes what generalises rather
than what the model memorised. The alternative — the tree ensemble's own split counts —
rewards high-cardinality features for being splittable and is not comparable across feature
types.

In [ ]:
# Counterfactual grid: an ordinary Wednesday, no holiday, no rain, with every
# continuous feature at its training mean **within that (airport, hour) cell**.
#
# Not at its overall mean. Four of the features are the demand history —
# `pickups_lag_1h` above all — and setting those to the average of the whole
# day asks the model what 06:00 would look like if the hour before it had been
# an average hour. No such 06:00 exists: the hour before it is 05:00, which is
# nearly empty. Held that way the profile puts JFK's peak at 06:00 and flattens
# the evening, contradicting the raw hourly averages in notebook 3, because the
# hour dummies are left carrying only the part of the daily cycle the lags do
# not already explain.
#
# Conditioning each feature on the hour keeps the schedule, the weather, and
# the history consistent with the hour being described, and what the panel then
# shows is the model's expectation for an ordinary midweek day — which is the
# question a driver is actually asking.
# Wednesday, and June rather than the median of the month index. The test
# period is January-June, so a figure the recommendations point at should
# describe a month inside it; and the median of a month *index* is not a month
# in any meaningful sense — it was reaching for a typical month and returning
# July, which the recommendations never cover. Either way the choice has to
# appear in the caption, so it is named here.
REFERENCE_DAY_OF_WEEK = 3
REFERENCE_MONTH = 6
REFERENCE_MONTH_NAME = "June"
assert REFERENCE_MONTH in set(train["month"]), (
    "The reference month is not present in the training split"
)

cell_means = glm_train.groupby(["airport", "hour"])[CONTINUOUS].mean()

profile_rows = []
for airport in AIRPORTS:
    for hour in range(24):
        row = dict(cell_means.loc[(airport, hour)])
        row.update({
            "hour": hour,
            "airport": airport,
            "day_of_week": REFERENCE_DAY_OF_WEEK,
            "month": REFERENCE_MONTH,
            "is_holiday": False,
            "is_holiday_adjacent": False,
            "is_precip": False,
            "is_freezing": False,
        })
        profile_rows.append(row)

profile = pd.DataFrame(profile_rows)
profile["fitted_rate"] = np.asarray(poisson_fit.predict(profile), dtype=float)

peak = profile.loc[profile.groupby("airport")["fitted_rate"].idxmax()]
trough = profile.loc[profile.groupby("airport")["fitted_rate"].idxmin()]
print(f"Expected demand, ordinary {REFERENCE_MONTH_NAME} midweek day, "
      "average conditions for the hour")
for airport in AIRPORTS:
    high = peak[peak["airport"] == airport].iloc[0]
    low = trough[trough["airport"] == airport].iloc[0]
    print(f"  {airport}: peak {high['fitted_rate']:6.1f} pickups at "
          f"{int(high['hour']):02d}:00, trough {low['fitted_rate']:5.1f} at "
          f"{int(low['hour']):02d}:00 "
          f"({high['fitted_rate'] / max(low['fitted_rate'], 0.01):.0f}x)")

importance = permutation_importance(
    gbt, x_value_test, y_value_test, n_repeats=5,
    random_state=SEED, scoring="neg_root_mean_squared_error",
)
importances = (
    pd.DataFrame({
        "feature": x_value_test.columns,
        "rmse_increase": importance.importances_mean,
        "sd": importance.importances_std,
    })
    .sort_values("rmse_increase", ascending=False)
    .reset_index(drop=True)
)
print("\nModel 2, permutation importance on the test split (US$ of RMSE)")
print(importances.head(10).round(3).to_string(index=False))

In [ ]:
# --- Figure 9 --------------------------------------------------------------
fig, axes = new_figure(1, 2, height=3.0)

ax = axes[0]
for airport in AIRPORTS:
    group = profile[profile["airport"] == airport]
    ax.plot(group["hour"], group["fitted_rate"], marker="o", markersize=3,
            color=airport_colour(airport), label=airport)
ax.set_xticks(range(0, 24, 3))
ax.set_xlabel("Hour of day")
ax.set_ylabel("Fitted pickups per hour")
ax.set_title(f"(a) Expected demand, midweek {REFERENCE_MONTH_NAME} day")
ax.legend()

ax = axes[1]
top = importances.head(10).iloc[::-1]
ax.barh(top["feature"], top["rmse_increase"], xerr=top["sd"],
        color=PALETTE["blue"], error_kw={"lw": 0.7, "ecolor": PALETTE["grey"]})
ax.set_xlabel("Increase in test RMSE when shuffled (US$)")
ax.set_title("(b) Model 2: permutation importance")
ax.grid(axis="y", visible=False)
ax.tick_params(axis="y", labelsize=7)

fig.tight_layout()
save_figure(fig, "fig09_model_interpretation")

## Step 6 — Combining the two models

Expected revenue in an airport-hour is the product of the two predictions:

`revenue = E[pickups] × E[fare per pickup]`

Notebook 2c verified the identity this rests on — `n_pickups × mean_total` reproduces
`sum_total_amount` to the cent — so the product estimates a quantity that exists in the data
rather than a plausible-looking composite.

Two properties of the combination are worth stating in the report.

**It repairs the support problem on both sides.** Model 2 is undefined in hours with no
pickups, which are overwhelmingly overnight, and the seasonal-naive fare forecast is
undefined in a further set of hours whose counterpart last week was empty — which is why
Step 4 has to score that comparison on a restricted set. The product is not: an hour with no trips earned nothing,
so its revenue is zero and is perfectly well defined. The combined prediction can therefore
be evaluated on **every** test hour, including the ones where the value model alone cannot be
scored, and the benchmark it is compared against is the product of the two naive forecasts.

**It is market revenue, not a wage.** `n_pickups × mean_total` is what every yellow taxi
collectively earned from that queue in that hour, not what one driver takes home. Turning it
into a driver's hourly rate needs the queue length, which the TLC data does not record —
trips are observed, waiting taxis are not. The honest reading is therefore comparative and
per-trip: the fare model says what a trip out of each queue is worth, and the demand model
says how fast that queue is clearing. Both are things a driver chooses between at the moment
of joining, and Step 7 states the assumption this rests on rather than hiding it.

In [ ]:
predictions["pred_revenue"] = (
    predictions["pred_pickups"] * predictions["pred_mean_total"]
)

# The seasonal-naive product, on the same basis: last week's count times last
# week's mean fare. Where last week's hour was empty its mean fare is undefined
# and its revenue was zero, which is what the product should be.
naive_revenue = np.where(
    predictions["pickups_lag_168h"].to_numpy() == 0,
    0.0,
    predictions["pickups_lag_168h"].to_numpy()
    * predictions["mean_total_lag_168h"].to_numpy(),
)
predictions["naive_revenue"] = naive_revenue

# Step 6 claims the combination is scorable on every test hour. Checked, not
# claimed: `sum_total_amount` is zero-filled in 2a for hours with no trips, and
# the naive product is zero wherever last week's hour was empty, so neither
# side has a null to drop.
assert predictions["sum_total_amount"].notna().all(), (
    "sum_total_amount is null somewhere; the revenue comparison would silently "
    "drop those hours"
)
assert not pd.isna(naive_revenue).any(), "The naive revenue forecast has nulls"

revenue_metrics = {
    "benchmark": error_metrics(
        predictions["sum_total_amount"], predictions["naive_revenue"]
    ),
    "combined": error_metrics(
        predictions["sum_total_amount"], predictions["pred_revenue"]
    ),
}
revenue_improvement = 100 * (
    1 - revenue_metrics["combined"]["rmse"] / revenue_metrics["benchmark"]["rmse"]
)

observed_total = float(predictions["sum_total_amount"].sum())
predicted_total = float(predictions["pred_revenue"].sum())

assert (
    revenue_metrics["benchmark"]["n"]
    == revenue_metrics["combined"]["n"]
    == len(predictions)
), "The revenue comparison is not scored on every test hour"

print(f"Expected revenue per airport-hour, all "
      f"{revenue_metrics['combined']['n']:,} test hours — both the benchmark "
      "and the combination are defined on every one of them")
print(f"  seasonal-naive product  RMSE ${revenue_metrics['benchmark']['rmse']:,.0f}   "
      f"MAE ${revenue_metrics['benchmark']['mae']:,.0f}")
print(f"  combined models         RMSE ${revenue_metrics['combined']['rmse']:,.0f}   "
      f"MAE ${revenue_metrics['combined']['mae']:,.0f}")
print(f"  improvement in RMSE: {revenue_improvement:+.1f}%")
print(f"\nTotal observed revenue over the test period:  ${observed_total:,.0f}")
print(f"Total predicted:                              ${predicted_total:,.0f}"
      f"  ({100 * (predicted_total / observed_total - 1):+.1f}%)")

In [ ]:
# The decision surface: what each queue is predicted to be worth, averaged over
# the six test months, by hour of day and day of week.
DAY_LABELS = ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"]
HOUR_LABELS = [f"{hour:02d}" for hour in range(24)]

surface = (
    predictions.groupby(["airport", "day_of_week", "hour"])
    .agg(
        pred_revenue=("pred_revenue", "mean"),
        pred_fare=("pred_mean_total", "mean"),
        pred_pickups=("pred_pickups", "mean"),
        observed_revenue=("sum_total_amount", "mean"),
    )
    .reset_index()
)


def surface_grid(frame: pd.DataFrame, airport: str, value: str) -> np.ndarray:
    """Reshape one airport's surface into a 7 × 24 grid, Monday first."""
    grid = (
        frame[frame["airport"] == airport]
        .pivot_table(index="day_of_week", columns="hour", values=value)
        .reindex(index=range(1, 8), columns=range(24))
    )
    return grid.to_numpy()


advantage = (
    surface_grid(surface, AIRPORTS[0], "pred_revenue")
    - surface_grid(surface, AIRPORTS[1], "pred_revenue")
)

hourly = (
    predictions.groupby(["airport", "hour"])
    .agg(
        predicted=("pred_revenue", "mean"),
        observed=("sum_total_amount", "mean"),
        fare=("pred_mean_total", "mean"),
    )
    .reset_index()
)

# --- Figure 10 -------------------------------------------------------------
fig, axes = new_figure(1, 2, height=3.0)

ax = axes[0]
for airport in AIRPORTS:
    group = hourly[hourly["airport"] == airport]
    ax.plot(group["hour"], group["predicted"], color=airport_colour(airport),
            label=f"{airport}, predicted")
    ax.plot(group["hour"], group["observed"], color=airport_colour(airport),
            ls="--", lw=1.0, alpha=0.7, label=f"{airport}, observed")
ax.set_xticks(range(0, 24, 3))
ax.set_xlabel("Hour of day")
ax.set_ylabel("Revenue per airport-hour")
ax.set_title("(a) Predicted against observed revenue")
currency_axis(ax)
ax.legend(fontsize=7)

ax = axes[1]
extreme = float(np.nanpercentile(np.abs(advantage), 98))
image = heatmap(
    ax, advantage, DAY_LABELS, HOUR_LABELS, cmap=DIVERGING_CMAP,
    vmin=-extreme, vmax=extreme,
    title=f"(b) {AIRPORTS[0]} minus {AIRPORTS[1]}, predicted revenue",
    xlabel="Hour of day", ylabel="Day of week",
)
bar = fig.colorbar(image, ax=ax, shrink=0.9, pad=0.02)
bar.set_label(f"{AIRPORTS[0]} advantage per hour", size=8)
currency_axis(bar.ax)

fig.tight_layout()
save_figure(fig, "fig10_expected_revenue")

## Step 7 — What this says to a driver

The two models answer a driver's two questions separately and then together, and the table
below is the bridge from the modelling section to the recommendations.

**The assumption the advice rests on, stated plainly.** The data records trips, not waiting
taxis, so no model here observes queue length. The comparison between two airports in the
same hour is therefore made under the assumption that the two queues are comparably supplied
— which is far weaker than it sounds, because both are fed by the same fleet under the same
regulator, and a driver's decision is precisely a choice between them at one moment. Where a
recommendation depends on the absolute wait rather than the comparison, it is not made.

In [ ]:
best_hours = (
    hourly.sort_values("predicted", ascending=False)
    .groupby("airport")
    .head(5)
    .sort_values(["airport", "predicted"], ascending=[True, False])
)
print("Highest predicted revenue per hour, by airport (test-period average)")
print(best_hours.round(1).to_string(index=False))

# Hours where the smaller airport is predicted to be worth more, which is the
# non-obvious half of the advice.
wide = hourly.pivot(index="hour", columns="airport", values="predicted")
fare_wide = hourly.pivot(index="hour", columns="airport", values="fare")
crossover = wide[wide[AIRPORTS[1]] > wide[AIRPORTS[0]]]
print(f"\nHours where {AIRPORTS[1]} is predicted to out-earn {AIRPORTS[0]}: "
      f"{list(crossover.index)}")
print("\nPredicted fare per trip, by hour (US$)")
print(fare_wide.round(2).to_string())

In [ ]:
# --- Persist ---------------------------------------------------------------
quasi_fit.save(str(MODELS_DIR / "poisson_glm_quasi.pickle"))
negbin_fit.save(str(MODELS_DIR / "negative_binomial_glm.pickle"))
joblib.dump(gbt, MODELS_DIR / "gbt_value.joblib")

# Coefficients with quasi-Poisson inference, exponentiated into rate ratios,
# which is the form the report quotes.
coefficients = pd.DataFrame({
    "term": quasi_fit.params.index,
    "coefficient": quasi_fit.params.to_numpy(),
    "std_error": quasi_fit.bse.to_numpy(),
    "p_value": quasi_fit.pvalues.to_numpy(),
    "rate_ratio": np.exp(quasi_fit.params.to_numpy()),
})
confidence = quasi_fit.conf_int()
coefficients["rate_ratio_low"] = np.exp(confidence[0].to_numpy())
coefficients["rate_ratio_high"] = np.exp(confidence[1].to_numpy())
coefficients.to_csv(MODELS_DIR / "glm_coefficients.csv", index=False)

KEEP = [
    "date", "hour", "airport", TARGET_VOLUME, TARGET_VALUE, "sum_total_amount",
    "pred_pickups", "pred_pickups_nb", "pred_mean_total", "pred_revenue",
    "naive_revenue", "residual",
]
predictions[KEEP].to_parquet(
    CURATED_DIR / "model_predictions.parquet", index=False
)

model_results = {
    "split": {
        "train_rows": int(len(train)),
        "test_rows": int(len(test)),
        "excluded_rows": int(excluded),
        "model_2_train_rows": int(len(value_train)),
        "model_2_test_rows": int(len(value_test)),
    },
    "model_1": {
        "family": "Poisson GLM, log link, quasi-Poisson scale",
        "parameters": int(poisson_fit.df_model) + 1,
        "pearson_dispersion": round(pearson_dispersion, 3),
        "negative_binomial_alpha": round(alpha, 4),
        "test": volume_metrics,
    },
    "model_2": {
        "family": "HistGradientBoostingRegressor",
        "hyperparameters": best,
        "validation_rmse": float(tuning.loc[0, "valid_rmse"]),
        "test": value_metrics,
        "top_features": importances.head(5)["feature"].tolist(),
    },
    "combined": {
        "test": revenue_metrics,
        "rmse_improvement_pct": round(revenue_improvement, 1),
        "observed_test_revenue": round(observed_total, 2),
        "predicted_test_revenue": round(predicted_total, 2),
    },
}
with open(CURATED_DIR / "model_results.json", "w") as handle:
    json.dump(model_results, handle, indent=2)

print(f"Models written to {MODELS_DIR}")
print(f"Predictions and metrics written to {CURATED_DIR}")
for path in sorted(PLOTS_DIR.glob("fig*.png")):
    print(f"  {path.name}")

## What to carry into the report

**Modelling section.** One paragraph per model, then one on the combination. For Model 1:
the family and link, the two assumptions that were tested rather than asserted
(overdispersion, independence), the quasi-Poisson correction, and the negative binomial
agreement. For Model 2: why a contrasting family rather than a second GLM, and that the
hyper-parameters were chosen on a forward validation split. For the combination: the
identity it rests on, and that it is evaluable on every test hour where the value model
alone is not.

**Quote the benchmark every time you quote an error.** A test RMSE alone is unanchored; the
same figure against *this hour will be what it was last week* is a result.

**Figure 9(a) is the figure the recommendations point at.** It is the daily demand shape with
the flight schedule, weather, and history held constant, which is a claim about the hour
itself rather than about everything correlated with it.

**Be explicit about what the revenue number is.** Market revenue per airport-hour, not a
driver's wage; the queue length is not in the data. The comparison between two airports at
the same moment is the defensible use of it, and it is enough to support the
recommendations.

**Report the error analysis, including where the model is biased, and what the bias is.**
The demand model's residuals are one-signed, and Step 5 shows this is attenuation rather
than a year-on-year level shift: the two move in opposite directions, so the level story
cannot be the explanation. Say so. Under-stating the busy hours and over-stating the quiet
ones is the direction that matters for the recommendations — it makes the peak advice
conservative and the overnight advice generous, and a driver should be told which way the
error leans.

**Quote errors with their row counts.** The value benchmark is undefined in hours whose
counterpart last week was empty, so the comparison table carries `n` per row and the two
value figures are not interchangeable.

### References to add to the bibliography

1. McCullagh, P. and Nelder, J. A. *Generalized Linear Models*, 2nd ed. Chapman and Hall,
   1989 — the Poisson GLM and the log link.
2. Cameron, A. C. and Trivedi, P. K. *Regression Analysis of Count Data*, 2nd ed. Cambridge
   University Press, 2013 — overdispersion, the quasi-Poisson scale, and the auxiliary
   regression used to estimate the negative binomial dispersion.
3. Friedman, J. H. "Greedy function approximation: a gradient boosting machine."
   *Annals of Statistics*, 29(5), 2001 — gradient boosting. Cite the scikit-learn
   implementation alongside it.

**Next:** the report itself. The numbers it quotes are in `data/curated/shapes_*.json`,
`analysis_findings.json`, and `model_results.json`; the figures are in `plots/`.